# Hankel Matrix Recovery

This notebook demonstrates recovery of an unknown **Hankel** matrix $A(\mathbf{c}^*)$ parameterized from a linearly-parameterized family:
$$A(\mathbf{c}) = \sum_{i=0}^{2n-2} \mathbf{c}(i)\, H_i$$
where $H_i$ are canonical Hankel basis matrices with constant anti-diagonals.

## Hankel Basis

The basis consists of $2n-1$ canonical Hankel matrices, each with a single 1 in one position
and zeros elsewhere. This directly parameterizes all Hankel matrices by their parameters.

In [1]:
import numpy as np
from scipy.linalg import hankel
from wias import CPQR_solver

rng = np.random.default_rng(1)

# Matrix dimension = basis size (2n-1 parameters for n×n Hankel)
n = 1000
N_basis = 2 * n - 1

## Generate Unknown Matrix

Create the unknown coefficient vector $\mathbf{c}^*$ (length $2n-1$)
and construct $A(\mathbf{c}^*) = \sum_{i=0}^{2n-2} c_i^* \, H_i$.

In [2]:
# Generate canonical Hankel basis matrices
# H_k has single 1 at one parameter position
basis_matrices = []

# First n basis matrices: correspond to first column
for k in range(n):
    first_col = np.zeros(n)
    first_col[k] = 1.0
    last_row = np.zeros(n)
    last_row[0] = 1.0  # Overlap element
    H_k = hankel(first_col, last_row)
    basis_matrices.append(H_k)

# Next n-1 basis matrices: correspond to last row (excluding overlap)
for k in range(1, n):
    first_col = np.zeros(n)
    first_col[0] = 1.0
    last_row = np.zeros(n)
    last_row[k] = 1.0
    H_k = hankel(first_col, last_row)
    basis_matrices.append(H_k)

In [3]:
# True coefficients (first column and last row parameters)
c_true = rng.standard_normal(N_basis)
A_true = sum(c * H_i for c, H_i in zip(c_true, basis_matrices))

## 1. Recover the Hankel matrix from one matrix-vector product

In [4]:
# Generate random input vectors
q_R = 1  # number of right inputs
q_L = 0  # number of left inputs

f_vecs = rng.standard_normal((q_R, n))
g_vecs = rng.standard_normal((q_L, n))

# Compute outputs
u_vecs = np.array([A_true @ f for f in f_vecs])
v_vecs = np.array([A_true.T @ g for g in g_vecs])

# Build linear recovery system
q_R, n_dim = f_vecs.shape
q_L, _ = g_vecs.shape

m_total = q_R * n_dim + q_L * n_dim
A_sys = np.zeros((m_total, N_basis))

for i, H_i in enumerate(basis_matrices):
    u_contrib = np.array([H_i @ f for f in f_vecs]).ravel()
    v_contrib = np.array([H_i.T @ g for g in g_vecs]).ravel()
    
    A_sys[:q_R*n_dim, i] = u_contrib
    A_sys[q_R*n_dim:, i] = v_contrib

b_sys = np.concatenate([u_vecs.ravel(), v_vecs.ravel()])

print(f"\nRecovery system : {A_sys.shape}")

c_recovered = CPQR_solver(A_sys, b_sys)

rec_error = np.linalg.norm(c_recovered - c_true)
print("-" * 50)
print(f"L2 recovery error: {rec_error:.2e}")


Recovery system : (1000, 1999)
--------------------------------------------------
L2 recovery error: 1.98e+02


## 2. Recover the Hankel matrix from 2 matrix-vector products

## Observe matrix-vector products

We observe:
- **Right** products: $u_j = A(\mathbf{c}^*) f_j$ for $j = 1, \ldots, q_R$
- **Left** products: $v_k = A(\mathbf{c}^*)^\top g_k$ for $k = 1, \ldots, q_L$

In [5]:
# Generate random input vectors
q_R = 2  # number of right inputs
q_L = 0  # number of left inputs

f_vecs = rng.standard_normal((q_R, n))
g_vecs = rng.standard_normal((q_L, n))

# Compute outputs
u_vecs = np.array([A_true @ f for f in f_vecs])
v_vecs = np.array([A_true.T @ g for g in g_vecs])

# Build linear recovery system
q_R, n_dim = f_vecs.shape
q_L, _ = g_vecs.shape

m_total = q_R * n_dim + q_L * n_dim
A_sys = np.zeros((m_total, N_basis))

for i, H_i in enumerate(basis_matrices):
    u_contrib = np.array([H_i @ f for f in f_vecs]).ravel()
    v_contrib = np.array([H_i.T @ g for g in g_vecs]).ravel()
    
    A_sys[:q_R*n_dim, i] = u_contrib
    A_sys[q_R*n_dim:, i] = v_contrib

b_sys = np.concatenate([u_vecs.ravel(), v_vecs.ravel()])

print(f"\nRecovery system : {A_sys.shape}")

c_recovered = CPQR_solver(A_sys, b_sys)

rec_error = np.linalg.norm(c_recovered - c_true)
print("-" * 50)
print(f"L2 recovery error: {rec_error:.2e}")


Recovery system : (2000, 1999)
--------------------------------------------------
L2 recovery error: 2.82e-11
